# Lab R1 — Ingestion & Parsing

**Curriculum §4 · Lab R1**

> *No downstream technique recovers information destroyed at parse time.*

You will render the Meridian corpus to real PDFs (including a genuine **table**),
then parse them with four parsers and score **table integrity** cell-by-cell.

| Compare | Measure |
|---|---|
| pypdf · pdfplumber · PyMuPDF · unstructured | text fidelity · table cell F1 · wall-time |


## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and pulls the shared `common/` modules.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade above + a restart.


In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())  # fix Colab PIL._typing._Ink mismatch
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu '
                   'rank_bm25 langchain langchain-community langchain-groq '
                   'langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF reportlab pandas'.split())

    # Pull the shared common/ package (config, corpus, golden, obs, scorers, harness).
    REPO = pathlib.Path('/content/genai_practical')
    if not (REPO/'common'/'harness.py').exists():
        # Option A: clone if you've pushed the repo to GitHub — set REPO_URL and uncomment:
        # subprocess.run(['git','clone','$REPO_URL', str(REPO)])
        # Option B: mount Google Drive where you unzipped the repo:
        try:
            from google.colab import drive; drive.mount('/content/drive')
            src = pathlib.Path('/content/drive/MyDrive/genai_practical')
            if src.exists(): REPO = src
        except Exception as e: print('Drive mount skipped:', e)
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e: print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))   # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `common` importable
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| docs in force:', [d['id'] for d in current_docs()])


## 1 · Generate the source PDFs
The rate card is rendered as a **real table** so parsers can genuinely fail on it.


In [ ]:
%pip install -q reportlab pypdf pdfplumber PyMuPDF


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

PDF_DIR = DATA / 'pdfs'; PDF_DIR.mkdir(exist_ok=True)
styles = getSampleStyleSheet()

RATE_ROWS = [['LTV','2-year fixed','Product fee'],
             ['60%','4.29%','GBP 1,495'],
             ['70%','4.55%','GBP 1,495'],
             ['75%','4.84%','GBP 1,495'],
             ['80%','Not offered','-']]

for d in DOCS:
    path = PDF_DIR / f"{d['id']}.pdf"
    doc  = SimpleDocTemplate(str(path), pagesize=A4)
    story = [Paragraph(f"{d['doc']} ({d['version']}) - effective {d['effective']}", styles['Title']),
             Spacer(1,12), Paragraph(d['text'], styles['BodyText'])]
    if d['id'] == 'rate-1':
        t = Table(RATE_ROWS)
        t.setStyle(TableStyle([('GRID',(0,0),(-1,-1),0.5,colors.grey),
                               ('BACKGROUND',(0,0),(-1,0),colors.lightgrey)]))
        story += [Spacer(1,16), t]
    doc.build(story)
print('wrote', len(DOCS), 'PDFs to', PDF_DIR)


## 2 · The parser shoot-out


In [ ]:
import time

def parse_pypdf(p):
    from pypdf import PdfReader
    return '\n'.join((pg.extract_text() or '') for pg in PdfReader(str(p)).pages)

def parse_pdfplumber(p):
    import pdfplumber
    out=[]
    with pdfplumber.open(str(p)) as pdf:
        for pg in pdf.pages:
            out.append(pg.extract_text() or '')
            for tbl in (pg.extract_tables() or []):
                out.append('\n'.join(' | '.join(c or '' for c in row) for row in tbl))
    return '\n'.join(out)

def parse_pymupdf(p):
    import fitz
    return '\n'.join(pg.get_text() for pg in fitz.open(str(p)))

PARSERS = {'pypdf':parse_pypdf, 'pdfplumber':parse_pdfplumber, 'pymupdf':parse_pymupdf}

results=[]
for name, fn in PARSERS.items():
    for d in DOCS:
        p = PDF_DIR / f"{d['id']}.pdf"
        t0=time.time()
        try: text, err = fn(p), None
        except Exception as e: text, err = '', f'{type(e).__name__}'
        results.append(dict(parser=name, doc=d['id'], ms=round((time.time()-t0)*1000,1),
                            chars=len(text), error=err, text=text))
import pandas as pd
df = pd.DataFrame(results)
df.groupby('parser')[['ms','chars']].mean().round(1)


## 3 · Score table integrity — the metric that matters

Character counts flatter bad parsers. What decides usefulness is whether the
**rate table survived**: can we still recover `75% -> 4.84%`?


In [ ]:
import re
TRUTH = {'60%':'4.29%','70%':'4.55%','75%':'4.84%','80%':'Not offered'}

def table_cell_recall(text):
    """Fraction of LTV->rate pairs still adjacent enough to be recoverable."""
    flat = re.sub(r'\s+',' ', text)
    ok = 0
    for ltv, rate in TRUTH.items():
        m = re.search(re.escape(ltv) + r'.{0,40}?' + re.escape(rate), flat)
        if m: ok += 1
    return ok/len(TRUTH)

rate = df[df.doc=='rate-1'].copy()
rate['table_recall'] = rate.text.apply(table_cell_recall)
rate[['parser','chars','ms','table_recall']].sort_values('table_recall', ascending=False)


## 4 · What you should be seeing

- Every parser returns *plenty of characters* for `rate-1` — they all look fine on volume.
- **`table_recall` separates them.** A parser that linearises the table column-wise
  destroys the LTV→rate adjacency, and no chunker or model can put it back.

### The architectural consequence
Choose parsers **per document type**, not globally:

| Doc type | Parser | Why |
|---|---|---|
| Policy prose | pypdf / PyMuPDF | fast, fidelity is enough |
| Rate cards, tables | pdfplumber (`extract_tables`) | preserves cell structure |
| Scanned packets | OCR (pytesseract / PaddleOCR) | no text layer exists |

**Record your decision in `results/` — Lab R2 builds on the winning parser.**

**Next →** `lab02_chunking.ipynb`
